In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from datetime import datetime
from wavelengths import read_wavelengths

In [2]:
def take_left(f, x, x_new):
    idx = np.searchsorted(x, x_new, side='right').clip(1, len(x))
    return np.take_along_axis(f, idx - 1, axis=0)

def get_mean_voltages(fg_voltages, fg_times, pmp_times):
    fg_voltages_ = take_left(fg_voltages, fg_times, pmp_times)
    return np.mean(fg_voltages_.reshape(-1,6,4,5), axis=(0,-1)).T

def get_bias(shifts):
    from scipy.special import voigt_profile
    from modulation import modulation_matrix
    from fit_pv import fit_pv

    q_B = 299792458 / 6173.341 * 0.231
    sigma = 0.043
    gamma = 0.053

    D = np.linalg.inv(modulation_matrix())
    F = []
    for shift in shifts:
        f = voigt_profile(shift, sigma, gamma)
        F += [f]
    F = D @ np.array(F)

    shift_ = np.round(np.mean(shifts, axis=0), 3)
    bias = fit_pv(F[0] + F[3], shift_)[0] - fit_pv(F[0] - F[3], shift_)[0]
    bias *= q_B
    return bias

In [3]:
files = sorted(glob.glob('/home/ulyanov/data/solo/phi/2026/data/*.fits.gz'))
files

['/home/ulyanov/data/solo/phi/2026/data/solo_L1_phi-fdt-alam_20260409T014503_V202604260832C_0644090501.fits.gz',
 '/home/ulyanov/data/solo/phi/2026/data/solo_L1_phi-fdt-alam_20260409T080003_V202606261130C_0644090503.fits.gz',
 '/home/ulyanov/data/solo/phi/2026/data/solo_L1_phi-fdt-alam_20260409T110003_V202607131830C_0644090504.fits.gz',
 '/home/ulyanov/data/solo/phi/2026/data/solo_L1_phi-fdt-alam_20260409T140003_V202607131930C_0644090505.fits.gz',
 '/home/ulyanov/data/solo/phi/2026/data/solo_L1_phi-fdt-alam_20260409T170003_V202607131930C_0644090506.fits.gz',
 '/home/ulyanov/data/solo/phi/2026/data/solo_L1_phi-fdt-alam_20260409T200003_V202607151630C_0644090507.fits.gz',
 '/home/ulyanov/data/solo/phi/2026/data/solo_L1_phi-fdt-alam_20260409T230003_V202607131730C_0644090508.fits.gz',
 '/home/ulyanov/data/solo/phi/2026/data/solo_L1_phi-fdt-alam_20260410T014503_V202604260832C_0644100501.fits.gz',
 '/home/ulyanov/data/solo/phi/2026/data/solo_L1_phi-fdt-alam_20260410T050003_V202604260932C_0644

In [4]:
file = files[10]

with fits.open(file) as hdul:
    header = hdul[0].header

    fg_data = hdul['PHI_FITS_FG_settings'].data
    fg_header = hdul['PHI_FITS_FG_settings'].header
    pmp_data = hdul['PHI_FITS_PMP_settings'].data
    pmp_header = hdul['PHI_FITS_PMP_settings'].header

fg_times = np.array([datetime.fromisoformat(temp) for temp in fg_data['RecordTime']])
fg_voltages = fg_data['PHI_FG_voltage'].astype(float)

pmp_times = np.array([datetime.fromisoformat(temp) for temp in pmp_data['RecordTime']])
pmp_voltages1 = pmp_data['PHI_PMP_FDT_voltage1'].astype(float)
pmp_voltages2 = pmp_data['PHI_PMP_FDT_voltage2'].astype(float)
pmp_states = pmp_data['PHI_PMP_FDT_state']

In [5]:
voltages = get_mean_voltages(fg_voltages, fg_times, pmp_times)
shifts = voltages * 3.513e-4 + (header['FGOV1PT1'] - 61) * 4.01225e-2 - header['OBS_VR'] * 6173.341 / 299792458
bias = get_bias(shifts)
bias

np.float64(-1.0172404213289052)

In [9]:
shifts = []

for file in files[:]:
    with fits.open(file) as hdul:
        fg_data = hdul['PHI_FITS_FG_settings'].data
        fg_header = hdul['PHI_FITS_FG_settings'].header
        pmp_data = hdul['PHI_FITS_PMP_settings'].data
        pmp_header = hdul['PHI_FITS_PMP_settings'].header

    fg_times = np.array([datetime.fromisoformat(temp) for temp in fg_data['RecordTime']])
    fg_voltages = fg_data['PHI_FG_voltage'].astype(float)

    pmp_times = np.array([datetime.fromisoformat(temp) for temp in pmp_data['RecordTime']])
    pmp_voltages1 = pmp_data['PHI_PMP_FDT_voltage1'].astype(float)
    pmp_voltages2 = pmp_data['PHI_PMP_FDT_voltage2'].astype(float)
    pmp_states = pmp_data['PHI_PMP_FDT_state']

    voltages = get_mean_voltages(fg_voltages, fg_times, pmp_times)
    shifts_ = voltages * 3.513e-4 + (header['FGOV1PT1'] - 61) * 4.01225e-2 - header['OBS_VR'] * 6173.341 / 299792458
    shifts += [shifts_]

shifts = np.array(shifts)
#np.savez('shifts.npz', shifts=shifts)

In [27]:
shifts = np.load('shifts.npz')['shifts']
shifts.shape

In [28]:
biases = np.array([get_bias(shifts_) for shifts_ in shifts])

In [29]:
from scipy.ndimage import gaussian_filter

plt.figure(figsize=(10,8))
plt.plot(biases, lw=0.5)
plt.plot(gaussian_filter(biases, 5), 'black', lw=2)
plt.ylim(-2,2)
plt.grid(True)
plt.tight_layout()